In [1]:
%cd /workspace
import os, json

#Modelo y cache de HF en el volumen
os.environ["HF_HOME"] = "/workspace/hf"
os.makedirs("/workspace/resultados", exist_ok=True)

/workspace


In [2]:
# vLLM, same version as training
!pip install -q uv pandas
!uv venv vllm --python 3.12
!uv pip install --python vllm/bin/python "vllm==0.33.0"

Using CPython 3.12.3 interpreter at: /usr/local/bin/python
Creating virtual environment at: vllm
Activate with: source vllm/bin/activate
Using Python 3.12.3 environment at: vllm
Resolved 199 packages in 980ms                                       
Prepared 199 packages in 10.76s                                          
Installed 199 packages in 1.68s                             
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.6.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==8.0.0
 + cbor2==6.2.1
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.8.6
 + cuda-python==13.5.0
 + cuda-tile==1.8.0
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect

In [5]:
%pip install -q huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [3]:
#FlashInfer kernels are disabled fix
#Si se atasca por la red, saltar esta celda y descomentar la linea de abajo
!./vllm/bin/vllm download-kernels
#os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

W1109 18:37:03.355000 859 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 18:37:03.356000 859 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 18:37:03.356000 859 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
FlashInfer kernels are disabled: flashinfer-cubin is not installed and nvcc (CUDA_HOME, CUDA_PATH, PATH or /usr/local/cuda) or ninja (PATH) is missing. Set CUDA_HOME to a CUDA toolkit and put ninja on PATH, or run `vllm download-kernels`.
=== Cubin Wheel Install ===
FlashInfer version: 0.7.2.post1
Wheel index: https://flashinfer.ai/whl
Requirement: flashinfer-cubin==0.7.2.post1
Command: /usr/local/bin/uv pip install --python /workspace/vllm/bin/python --upgrade --no-deps --index-url https://flashinfer.ai/whl flashinfer-cubin==0.7.2.post1
Using Python 3.12.3 environment at: vllm
Resolved 1 package in 510ms                

In [6]:
from huggingface_hub import HfApi

#Each epoch upload is a commit, copy the hash I want
for commit in HfApi().list_repo_commits("gabmerlo/salamandra-7b-eagle3-alia"):
    print(commit.created_at.strftime("%H:%M"), commit.commit_id, commit.title)

17:38 381ed00bab0e036b3e5b977a608afab877e3a749 Upload folder using huggingface_hub
17:12 830f1fa96c190bbea10e85866e439dd342e0bb4d Upload folder using huggingface_hub
16:50 a05f3127df3f93a74a06fbe667aebbdf521b589f Upload folder using huggingface_hub
16:50 de20c4c3e37a17fad9fbbdfbaea6f2a09e8c18b2 initial commit


In [7]:
#El del epoch 0 empieza por a05f312, el epoch 2 es el ultimo que subi
CHECKPOINTS = {
    "eagle3_ep0": "a05f3127df3f93a74a06fbe667aebbdf521b589f",
    "eagle3_ep2": "381ed00bab0e036b3e5b977a608afab877e3a749",
}

#Without the optimizer state, only what vLLM needs
for etiqueta, revision in CHECKPOINTS.items():
    !./vllm/bin/hf download gabmerlo/salamandra-7b-eagle3-alia --revision {revision} --local-dir /workspace/{etiqueta} --exclude "optimizer_state_dict.pt" "scheduler_state_dict.pt" "train.log"

!./vllm/bin/hf download BSC-LT/salamandra-7b-instruct-2606

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
/workspace/vllm/lib/python3.12/site-packages/huggingface_hub/cli/download.py:160: UserWarning: Ignoring `--exclude` since filenames have been explicitly set.
  warnings.warn("Ignoring `--exclude` since filenames have been explicitly set.")
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 2 files:   0%|                                  | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 29.9MB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 29.9MB            


Fetching 2 files: 100%|██████████████████████████| 2/2 [00:03<00:00,  1.73s/it]
Download complete: ████████████████████████████████████████| 3.02MB, 3.32MB/s  
Reconstruction complete: 100%|████████████████████| 29.9MB / 29.9MB, 12.4MB/s  ✓ Downloaded
  path: /workspace/eagle3_ep0
Download complete: █████████████████████

In [8]:
#Cuantas prompts por idioma y cuanto puede generar cada respuesta
N_POR_IDIOMA = 20
MAX_TOKENS = 512

def prompt_de(conversacion):
    #Only the messages before the first assistant answer
    mensajes = []
    for mensaje in conversacion["messages"]:
        if mensaje["role"] == "assistant":
            break
        mensajes.append({"role": mensaje["role"], "content": mensaje["content"]})
    return mensajes

prompts = {"es": [], "ca": [], "eu": [], "gl": []}

with open("/workspace/data/alia_eval_prompts.jsonl", encoding="utf-8") as archivo:
    for linea in archivo:
        conversacion = json.loads(linea)
        idioma = conversacion["lang"]
        if idioma in prompts and len(prompts[idioma]) < N_POR_IDIOMA:
            prompts[idioma].append(prompt_de(conversacion))

print({idioma: len(lista) for idioma, lista in prompts.items()})

{'es': 20, 'ca': 20, 'eu': 20, 'gl': 20}


In [9]:
import subprocess, signal, time, re
import requests

URL = "http://127.0.0.1:8000"

def arrancar(modelo, log):
    #Sin prefix caching para que no influya en la medida
    with open(log, "w") as archivo:
        servidor = subprocess.Popen(
            [
                "./vllm/bin/vllm", "serve", modelo,
                "--max-model-len", "8192",
                "--port", "8000",
                "--no-enable-prefix-caching"
            ],
            stdout=archivo,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    #Wait until it answers
    while True:
        if servidor.poll() is not None:
            raise RuntimeError("vLLM se ha parado, mirar " + log)
        try:
            requests.get(URL + "/v1/models", timeout=5)
            break
        except Exception:
            time.sleep(10)

    nombre = requests.get(URL + "/v1/models").json()["data"][0]["id"]
    print("vLLM listo:", nombre)
    return servidor, nombre

def parar(servidor):
    os.killpg(servidor.pid, signal.SIGTERM)
    servidor.wait()
    time.sleep(15)

def metricas_spec():
    #Speculative decoding counters from /metrics (only exist with the draft)
    valores = {}
    for linea in requests.get(URL + "/metrics").text.splitlines():
        if not linea.startswith("vllm:spec_decode") or "_created" in linea:
            continue
        nombre_completo, valor = linea.rsplit(" ", 1)
        nombre = nombre_completo.split("{")[0]
        posicion = re.search(r'position="(\d+)"', nombre_completo)
        if posicion:
            nombre = nombre + "[" + posicion.group(1) + "]"
        valores[nombre] = valores.get(nombre, 0) + float(valor)
    return valores

In [10]:
from concurrent.futures import ThreadPoolExecutor

def pedir(modelo, mensajes):
    #Temperatura 0 para poder comparar
    respuesta = requests.post(URL + "/v1/chat/completions", json={
        "model": modelo,
        "messages": mensajes,
        "temperature": 0,
        "max_tokens": MAX_TOKENS
    }, timeout=900).json()
    return {
        "texto": respuesta["choices"][0]["message"]["content"],
        "tokens": respuesta["usage"]["completion_tokens"]
    }

def evaluar(etiqueta, modelo, concurrencia):
    #Warmup, so CUDA graphs don't count
    pedir(modelo, prompts["es"][0])
    pedir(modelo, prompts["ca"][0])

    filas = []
    for idioma, lista in prompts.items():
        antes = metricas_spec()
        inicio = time.time()
        with ThreadPoolExecutor(max_workers=concurrencia) as grupo:
            resultados = list(grupo.map(lambda m: pedir(modelo, m), lista))
        segundos = time.time() - inicio
        despues = metricas_spec()

        diferencia = {k: despues.get(k, 0) - antes.get(k, 0) for k in despues}
        borradores = diferencia.get("vllm:spec_decode_num_drafts_total", 0)
        propuestos = diferencia.get("vllm:spec_decode_num_draft_tokens_total", 0)
        aceptados = diferencia.get("vllm:spec_decode_num_accepted_tokens_total", 0)
        tokens = sum(r["tokens"] for r in resultados)

        fila = {
            "modo": etiqueta,
            "concurrencia": concurrencia,
            "idioma": idioma,
            "tokens": tokens,
            "segundos": round(segundos, 1),
            "tok_s": round(tokens / segundos, 1),
            "longitud_aceptacion": round(1 + aceptados / borradores, 2) if borradores else None,
            "tasa_aceptacion": round(aceptados / propuestos, 3) if propuestos else None,
            "por_posicion": {k.split("[")[1][:-1]: v for k, v in diferencia.items() if "per_pos" in k},
            "textos": [r["texto"] for r in resultados]
        }
        filas.append(fila)
        print(etiqueta, "| c =", concurrencia, "|", idioma, "|", fila["tok_s"], "tok/s | aceptacion:", fila["longitud_aceptacion"])
    return filas

In [14]:
#1) Salamandra sola
servidor, modelo = arrancar("BSC-LT/salamandra-7b-instruct-2606", "/workspace/vllm_base.log")

resultados = []
resultados += evaluar("base", modelo, 1)
resultados += evaluar("base", modelo, 8)

parar(servidor)

vLLM listo: BSC-LT/salamandra-7b-instruct-2606
base | c = 1 | es | 166.8 tok/s | aceptacion: None
base | c = 1 | ca | 166.9 tok/s | aceptacion: None
base | c = 1 | eu | 167.1 tok/s | aceptacion: None
base | c = 1 | gl | 167.2 tok/s | aceptacion: None
base | c = 8 | es | 914.6 tok/s | aceptacion: None
base | c = 8 | ca | 826.9 tok/s | aceptacion: None
base | c = 8 | eu | 887.4 tok/s | aceptacion: None
base | c = 8 | gl | 1112.0 tok/s | aceptacion: None


In [12]:
#Que prompts fallan y por que
for idioma, lista in prompts.items():
    for i, mensajes in enumerate(lista):
        r = requests.post(URL + "/v1/chat/completions", json={
            "model": modelo, "messages": mensajes, "max_tokens": 1
        }).json()
        if "choices" not in r:
            print(idioma, i, str(r)[:300])

es 13 {'error': {'message': "This model's maximum context length is 8192 tokens. However, your prompt contains at least 8193 input tokens, which exceeds this limit even without any output tokens. Please reduce the length of the input prompt. (parameter=input_tokens, value=8193)", 'type': 'BadRequestError'


In [13]:
#Only prompts that fit with MAX_TOKENS of answer
def cabe(mensajes):
    r = requests.post(URL + "/tokenize", json={"model": modelo, "messages": mensajes}).json()
    return r["count"] + MAX_TOKENS <= 8192

prompts = {"es": [], "ca": [], "eu": [], "gl": []}

with open("/workspace/data/alia_eval_prompts.jsonl", encoding="utf-8") as archivo:
    for linea in archivo:
        conversacion = json.loads(linea)
        idioma = conversacion["lang"]
        if idioma in prompts and len(prompts[idioma]) < N_POR_IDIOMA:
            mensajes = prompt_de(conversacion)
            if cabe(mensajes):
                prompts[idioma].append(mensajes)

print({idioma: len(lista) for idioma, lista in prompts.items()})

{'es': 20, 'ca': 20, 'eu': 20, 'gl': 20}


In [15]:
#2) Salamandra + EAGLE-3 (el checkpoint ya dice que modelo grande usar)
for etiqueta in CHECKPOINTS:
    servidor, modelo = arrancar("/workspace/" + etiqueta, "/workspace/vllm_" + etiqueta + ".log")
    resultados += evaluar(etiqueta, modelo, 1)
    resultados += evaluar(etiqueta, modelo, 8)
    parar(servidor)

vLLM listo: BSC-LT/salamandra-7b-instruct-2606
eagle3_ep0 | c = 1 | es | 166.1 tok/s | aceptacion: None
eagle3_ep0 | c = 1 | ca | 166.8 tok/s | aceptacion: None
eagle3_ep0 | c = 1 | eu | 166.8 tok/s | aceptacion: None
eagle3_ep0 | c = 1 | gl | 167.7 tok/s | aceptacion: None
eagle3_ep0 | c = 8 | es | 916.1 tok/s | aceptacion: None
eagle3_ep0 | c = 8 | ca | 813.9 tok/s | aceptacion: None
eagle3_ep0 | c = 8 | eu | 937.3 tok/s | aceptacion: None
eagle3_ep0 | c = 8 | gl | 1095.9 tok/s | aceptacion: None
vLLM listo: BSC-LT/salamandra-7b-instruct-2606
eagle3_ep2 | c = 1 | es | 166.4 tok/s | aceptacion: None
eagle3_ep2 | c = 1 | ca | 166.8 tok/s | aceptacion: None
eagle3_ep2 | c = 1 | eu | 167.3 tok/s | aceptacion: None
eagle3_ep2 | c = 1 | gl | 167.1 tok/s | aceptacion: None
eagle3_ep2 | c = 8 | es | 931.7 tok/s | aceptacion: None
eagle3_ep2 | c = 8 | ca | 855.7 tok/s | aceptacion: None
eagle3_ep2 | c = 8 | eu | 868.6 tok/s | aceptacion: None
eagle3_ep2 | c = 8 | gl | 972.8 tok/s | aceptacion

In [ ]:
import pandas as pd

tabla = pd.DataFrame([{k: v for k, v in fila.items() if k not in ("textos", "por_posicion")} for fila in resultados])

#Speedup = tok/s EAGLE-3 / tok/s base, mismo idioma y concurrencia
base = tabla[tabla.modo == "base"].set_index(["concurrencia", "idioma"])["tok_s"]

for etiqueta in CHECKPOINTS:
    eagle = tabla[tabla.modo == etiqueta].set_index(["concurrencia", "idioma"])
    eagle = eagle.assign(tok_s_base=base, speedup=(eagle["tok_s"] / base).round(2))
    print(eagle[["tok_s_base", "tok_s", "speedup", "longitud_aceptacion", "tasa_aceptacion"]])
    print()

===== eagle3_ep0 =====
                     tok_s_base   tok_s  speedup  longitud_aceptacion  \
concurrencia idioma                                                     
1            es           166.8   282.7     1.69                 2.03   
             ca           166.9   312.5     1.87                 2.24   
             eu           167.1   298.0     1.78                 2.13   
             gl           167.2   296.5     1.77                 2.11   
8            es           914.6  1443.8     1.58                 2.02   
             ca           826.9  1537.0     1.86                 2.21   
             eu           887.4  1509.4     1.70                 2.06   
             gl          1112.0  1748.1     1.57                 2.12   

                     tasa_aceptacion  
concurrencia idioma                   
1            es                0.342  
             ca                0.414  
             eu                0.376  
             gl                0.369  
8           

In [26]:
#Aceptacion por posicion del draft (1o, 2o, 3er token), con 1 peticion a la vez
for fila in resultados:
    if fila["modo"] != "base" and fila["concurrencia"] == 1:
        print(fila["modo"], fila["idioma"], fila["por_posicion"])

eagle3_ep0 es {'0': 1547.0, '1': 862.0, '2': 483.0}
eagle3_ep0 ca {'0': 1450.0, '1': 907.0, '2': 539.0}
eagle3_ep0 eu {'0': 1451.0, '1': 862.0, '2': 513.0}
eagle3_ep0 gl {'0': 1509.0, '1': 895.0, '2': 503.0}
eagle3_ep2 es {'0': 1579.0, '1': 1039.0, '2': 676.0}
eagle3_ep2 ca {'0': 1451.0, '1': 1028.0, '2': 734.0}
eagle3_ep2 eu {'0': 1476.0, '1': 996.0, '2': 681.0}
eagle3_ep2 gl {'0': 1538.0, '1': 1034.0, '2': 710.0}


In [27]:
#A temperatura 0 las respuestas deberian ser casi todas iguales
for etiqueta in CHECKPOINTS:
    for idioma in prompts:
        textos_base = next(f["textos"] for f in resultados if f["modo"] == "base" and f["concurrencia"] == 1 and f["idioma"] == idioma)
        textos_eagle = next(f["textos"] for f in resultados if f["modo"] == etiqueta and f["concurrencia"] == 1 and f["idioma"] == idioma)
        iguales = sum(a == b for a, b in zip(textos_base, textos_eagle))
        print(etiqueta, idioma, "identicas:", iguales, "/", len(textos_base))

eagle3_ep0 es identicas: 5 / 20
eagle3_ep0 ca identicas: 8 / 20
eagle3_ep0 eu identicas: 10 / 20
eagle3_ep0 gl identicas: 5 / 20
eagle3_ep2 es identicas: 5 / 20
eagle3_ep2 ca identicas: 8 / 20
eagle3_ep2 eu identicas: 10 / 20
eagle3_ep2 gl identicas: 5 / 20


In [28]:
#Guardo todo y lo subo a HF (eval/) antes de terminar el pod
with open("/workspace/resultados/eval_checkpoints.json", "w", encoding="utf-8") as archivo:
    json.dump(resultados, archivo, ensure_ascii=False, indent=2)

tabla.to_csv("/workspace/resultados/eval_checkpoints.csv", index=False)

!cp /workspace/vllm_*.log /workspace/resultados/
!./vllm/bin/hf upload gabmerlo/salamandra-7b-eagle3-alia /workspace/resultados eval --private

Found 5 files to upload
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
Error: (Request ID: Root=1-6af21c47-5dcbe9a63e8ce88962045b8a;1953a959-c1c0-48ba-a7ed-8979b3df6711)

Bad request for commit endpoint:
Your push was rejected because it contains binary files. Please use https://huggingface.co/docs/hub/xet to store binary files. See also: https://huggingface.co/docs/hub/xet/using-xet-storage#git Offending files: - eval/vllm_base.log (ref: refs/heads/main)
Your push was rejected because it contains binary files. Please use https://huggingface.co/docs/hub/xet to store binary files. See also: https://huggingface.co/docs/hub/xet/using-xet-storage#git  Offending files:   

In [20]:
#Did vLLM load the draft?
!grep -o "speculative_config=[^,]*" /workspace/vllm_eagle3_ep2.log | head -1
!ls /workspace/eagle3_ep2
!head -c 1500 /workspace/eagle3_ep2/config.json

scheduler_state_dict.pt  train.log
head: cannot open '/workspace/eagle3_ep2/config.json' for reading: No such file or directory


In [24]:
#Quito las filas de EAGLE-3 que no valen, la base se queda
resultados = [f for f in resultados if f["modo"] == "base"]

for etiqueta in CHECKPOINTS:
    log = "/workspace/vllm_" + etiqueta + ".log"
    spec = json.dumps({"method": "eagle3", "model": "/workspace/" + etiqueta, "num_speculative_tokens": 3})

    with open(log, "w") as archivo:
        servidor = subprocess.Popen(
            [
                "./vllm/bin/vllm", "serve", "BSC-LT/salamandra-7b-instruct-2606",
                "--speculative-config", spec,
                "--max-model-len", "8192",
                "--port", "8000",
                "--no-enable-prefix-caching"
            ],
            stdout=archivo,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    while True:
        if servidor.poll() is not None:
            raise RuntimeError("vLLM se ha parado, mirar " + log)
        try:
            requests.get(URL + "/v1/models", timeout=5)
            break
        except Exception:
            time.sleep(10)

    #Esta vez tiene que salir algo distinto de None
    !grep -o "speculative_config=[^,]*" {log} | head -1

    modelo = "BSC-LT/salamandra-7b-instruct-2606"
    resultados += evaluar(etiqueta, modelo, 1)
    resultados += evaluar(etiqueta, modelo, 8)
    parar(servidor)

speculative_config=SpeculativeConfig(method='eagle3'
eagle3_ep0 | c = 1 | es | 282.7 tok/s | aceptacion: 2.03
eagle3_ep0 | c = 1 | ca | 312.5 tok/s | aceptacion: 2.24
eagle3_ep0 | c = 1 | eu | 298.0 tok/s | aceptacion: 2.13
eagle3_ep0 | c = 1 | gl | 296.5 tok/s | aceptacion: 2.11
eagle3_ep0 | c = 8 | es | 1443.8 tok/s | aceptacion: 2.02
eagle3_ep0 | c = 8 | ca | 1537.0 tok/s | aceptacion: 2.21
eagle3_ep0 | c = 8 | eu | 1509.4 tok/s | aceptacion: 2.06
eagle3_ep0 | c = 8 | gl | 1748.1 tok/s | aceptacion: 2.12
speculative_config=SpeculativeConfig(method='eagle3'
eagle3_ep2 | c = 1 | es | 317.8 tok/s | aceptacion: 2.36
eagle3_ep2 | c = 1 | ca | 359.0 tok/s | aceptacion: 2.6
eagle3_ep2 | c = 1 | eu | 339.7 tok/s | aceptacion: 2.45
eagle3_ep2 | c = 1 | gl | 341.6 tok/s | aceptacion: 2.46
eagle3_ep2 | c = 8 | es | 1639.4 tok/s | aceptacion: 2.31
eagle3_ep2 | c = 8 | ca | 1857.2 tok/s | aceptacion: 2.57
eagle3_ep2 | c = 8 | eu | 1880.5 tok/s | aceptacion: 2.41
eagle3_ep2 | c = 8 | gl | 1960.3 

In [22]:
!pkill -f "vllm serve"
time.sleep(20)
!nvidia-smi --query-gpu=memory.used --format=csv

memory.used [MiB]
0 MiB


In [23]:
#Un solo patron, asi --exclude no se come los demas
for etiqueta, revision in CHECKPOINTS.items():
    !rm -rf /workspace/{etiqueta}
    !./vllm/bin/hf download gabmerlo/salamandra-7b-eagle3-alia --revision {revision} --local-dir /workspace/{etiqueta} --exclude "*_state_dict.pt"
    !ls /workspace/{etiqueta}

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 8 files:   0%|                                  | 0/8 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/eagle3_ep0/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0%|       |  0.00B / 29.9MB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 2.90GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 2.90GB            

Fetching 8 files:  12%|███▎                      | 1/8 [00:00<00:03,  2.22it/s]
Reconstructing (incomplete total...):   0%|       |   512B / 2.90GB            
Reconstructing (incomplete total...):   0%|       | 2.42kB / 2.90GB            
Reconstructing (incomplete total...):   0%|       | 2.81kB / 2.90GB            
Reconstructing (incomplete total...):   0%|       | 2.85kB / 2.90GB            

Fetching 8 files:  62%|████████████████▎         | 5/8 [00:00<00:00, 11.28it/s]
Reco

In [29]:
#Logs fuera: HF los toma por binarios. --delete borra tambien los logs viejos de eval/
!rm /workspace/resultados/vllm_*.log
!./vllm/bin/hf upload gabmerlo/salamandra-7b-eagle3-alia /workspace/resultados eval --private --delete "*.log"

Found 2 files to upload
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ████████████████████  2 / 2 ✓
✓ Uploaded
  url: https://huggingface.co/gabmerlo/salamandra-7b-eagle3-alia/commit/a6e162549090dec937efd3f4b58460e51098ada4
